# khalij-watch: train the stride-2 SAR point detector

Trains `training.model.PointDetector` (timm EfficientNet encoder, ImageNet weights, U-Net decoder, objectness + offset heads) on the private `khalij-watch-chips` dataset, then exports ONNX.

Settings: **GPU (P100 or T4)** and **Internet on** (needs a phone-verified account) for the ImageNet weights and `onnx`/`onnxruntime`.

Checkpoints go to `/kaggle/working/run` every epoch. If the session stops (12 h cap), re-run: training resumes from `last.pt`. Save the notebook version to keep `/kaggle/working`.

Labels: Global Fishing Watch (non-commercial). Contains modified Copernicus Sentinel data.

In [ ]:
import subprocess
import sys
from pathlib import Path

DATA = next(p.parent for p in Path('/kaggle/input').rglob('index.jsonl'))
sys.path.insert(0, str(DATA / 'code'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'], check=True)

import timm  # noqa: E402 - after the path setup above
import torch  # noqa: E402

print('data', DATA)
print('torch', torch.__version__, 'timm', timm.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from training.train import Config, train

config = Config(
    index=str(DATA / 'index.jsonl'),
    root=str(DATA),
    out='/kaggle/working/run',
    encoder='efficientnet_b0',
    pretrained=True,
    objectness_loss='reduced_focal',
    epochs=15,
    samples_per_epoch=40_000,
    batch_size=64,
    lr=1e-3,
    workers=4,
)
final = train(config)
final

In [ ]:
import json

from training.export import export_onnx, load_model, max_difference, quantise

model = load_model(Path('/kaggle/working/run/best.pt'))
fp32 = export_onnx(model, Path('/kaggle/working/point_detector.onnx'))
int8 = quantise(fp32)
report = {'fp32_max_logit_diff': max_difference(model, fp32), 'int8_max_logit_diff': max_difference(model, int8)}
Path('/kaggle/working/export.json').write_text(json.dumps(report, indent=1))
print(json.dumps(report, indent=1))
print(json.dumps(json.loads(Path('/kaggle/working/run/history.json').read_text())[-3:], indent=1))